### Results from last week with no layers frozen

```python
{'test_loss': 0.28392818570137024,
 'test_precision': 0.5798479087452472,
 'test_recall': 0.28266913809082483,
 'test_f1': 0.38006230529595014,
 'test_accuracy': 0.942242742935317,
 'test_runtime': 5.4759,
 'test_samples_per_second': 235.029,
 'test_steps_per_second': 3.835}
 ```

### PEFT run #1
training_args:
```python
training_args = TrainingArguments(
    output_dir='./model_output',    # Directory where model checkpoints and outputs will be saved.
    num_train_epochs=3,# Total number of training epochs.
    per_device_train_batch_size=16, # Batch size per device during training.
    per_device_eval_batch_size=64,  # Batch size for evaluation.
    learning_rate=5e-5,             # Learning rate
    warmup_steps=500,               # Number of warmup steps for learning rate scheduler.
    weight_decay=0.01,              # Weight decay if we apply some.
    logging_dir='./logs',           # Directory for storing logs.
    logging_steps=10,               # Log every X updates steps.
    eval_strategy="steps",    # Evaluate every X steps.
    eval_steps=50,                  # Number of steps to evaluate after.
    save_strategy="steps",          # The checkpoint save strategy to use.
    save_steps=100,                 # Save checkpoint every X steps.
    load_best_model_at_end=True,    # Whether to load the best model found at each evaluation.
    report_to="none",                # The list of integrations to report the results and logs to.
    metric_for_best_model="f1",       # use F1 to decide what "best" means
    greater_is_better=True           # higher F1 is better (opposite of loss)
)
```
- Using LoRA with above parameters
```python
{'test_loss': 0.37263011932373047,
 'test_precision': 0.29347826086956524,
 'test_recall': 0.025023169601482854,
 'test_f1': 0.04611443210930828,
 'test_accuracy': 0.9272797229703732,
 'test_runtime': 5.8427,
 'test_samples_per_second': 220.275,
 'test_steps_per_second': 3.594}
 ```

# Part 1: PEFT Finetuning

In this part, we'll learn to finetune a pretrained BERT-Style (Encoder-only) transformer model for sequence classification task using LoRA and Adapters, two leading parameter-efficient finetuning techniques. We'll walk you through how these techniques are implemented from scratch, how to attach them to the pretrained language models, and to train them. We will explain them in details in their corresponding sections below. For detailed information, refer to the Week 8 lecture slides and the references given there.

This Lab uses the dataset used in [HuggingFace Token Classification Tutorial](https://huggingface.co/docs/transformers/v4.17.0/en/tasks/token_classification).




![](https://drive.google.com/uc?export=view&id=1eH5hr0WtFX8OhSknkQOhd1yR_V_9p3PV)


Image Source: Sabry & Belz, 2023, https://arxiv.org/abs/2304.12410



In [1]:
# Import the clear_output function from IPython.display module for clearing Jupyter Notebook cell output
from IPython.display import clear_output
# Install the necessary libraries for handling datasets and tokenization
# Tokenizers library from Hugging Face, specifically for creating and training custom tokenizers
!pip install transformers tokenizers accelerate evaluate -U seqeval ipywidgets
!pip install -U "datasets<4.0.0"
clear_output()

In [2]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score

from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers import BertModel, BertTokenizerFast
from transformers import TrainingArguments
from transformers import Trainer

from transformers import DataCollatorForTokenClassification

import evaluate
from datasets import load_dataset

from tqdm import tqdm

import time

from typing import List, Optional, Tuple, Union

import ipywidgets as widgets
from IPython.display import display


# Check if CUDA (GPU acceleration) is available on the system
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

# set a model name as in HuggingFace Model Hub, we will use to load the model weights.
# in this lab we will use a small size pretrained model (BERT-base)
pretrained_model_name = 'bert-base-uncased'

## 1 Extend Pretrained BERT with a Classifier

In [3]:
class BertClassifier(nn.Module):
    def __init__(self, bert_model: BertModel, num_labels: int):
        super(BertClassifier, self).__init__()
        """
          Parameters:
            - bert_model (BertModel): Pretrained BERT model to initialise our classifier.
            - num_labels (int): The number of labels for our classifier's output.

          Attributes:
            - bert (BertModel): BERT model.
            - num_labels (int): Number of labels in output.
            - classifier (nn.Linear): Linear transformation for the output of the
                pretrained model to get the labels prediction.
            - dropout (nn.Dropout): Dropout layer to prevent overfitting.
        """

        # use the model given in input to the init function, which is a pretrained BERT model
        self.bert = bert_model

        # Number of labels in the sequence classification task
        self.num_labels = num_labels

        # We can access the information like hidden size, vocab, #layers, ect..
        # of a pretrained model in its config structure, which is accessible at model.config
        # https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertConfig

        # Classifier layer to map the output of BERT to label space
        self.classifier = nn.Linear(bert_model.config.hidden_size, num_labels)

        # Dropout layer to prevent overfitting
        # here we used the same dropout value as the model hidden states dropout
        self.dropout = nn.Dropout(bert_model.config.hidden_dropout_prob)


    def forward(self, input_ids: torch.Tensor,
                attention_mask: torch.Tensor,
                token_type_ids: torch.Tensor,
                labels: torch.Tensor=None):
        """
        Forward pass for the BERT-based classifier.

        Parameters:
        - input_ids (torch.Tensor): tensor containing the ids of the tokenised input.
        - attention_mask (torch.Tensor): tensor containing the attention mask of the tokenised input.
        - token_type_ids (torch.Tensor): tensor containing the token type ids of the tokenised input.
        - labels (torch.Tensor): tensor containing the aligned target labels.

        Returns:
        - torch.Tensor: Output of the classifier.
        """

        # Get the outputs from pretrained BERT
        outputs = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)

        # we will use the last hidden state information, since it's an accumulation of processed information in previous layers
        # Some practitioner, instead of the last hidden state, they take all layers hidden states and average them.
        # For many tasks last hidden state works the best.
        # BERT model also return all hidden states and you can access it using outputs.hidden_states
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        output = (logits,) + outputs[2:]
        return ((loss,) + output) if loss is not None else output


## 2 Prepare Finetuning Dataset


For more information on how to prepare a dataset check this [YouTube tutorial](https://youtu.be/iY2AZYdZAr0).

To fine-tune our BERT model, we use [wnut_17 dataset](https://huggingface.co/datasets/wnut_17).

This dataset focuses on the task of "Emerging and Rare Entity Recognition." It is particularly aimed at identifying unusual, previously unseen entities within the context of emerging discussions. The recall of these entities can be challenging, especially when they are rare or emerging in nature.

It is composed of *tokens* and respective *NER tags*, which are expressed as int values that describe entities, such as corporation, location, or person.

Each tag value is associated to a specific tag:

- 0: O
- 1: B-corporation
- 2: I-corporation
- 3: B-creative-work
- 4: I-creative-work
- 5: B-group
- 6: I-group
- 7: B-location
- 8: I-location
- 9: B-person
- 10: I-person
- 11: B-product
- 12: I-product

NER tags are represented in **IOB format** in which the letter that prefixes each tag indicates the token position of the entity:

- B: beginning of the entity;
- I: (Inside) a token is contained inside the same entity;
- O: the token doesn't correspond to any entity.

The dataset is divided in train, validation and test set as follows:


Split | Size
------|-----
Train | 3394
Validation  | 1009
Test  | 1287

In [4]:
wnut = load_dataset("wnut_17")  # Load the dataset from HuggingFace

In [5]:
# Checking the first element in the dataset, we can see that each token has an associated NER tag expressed as int values
wnut["train"][0]

{'id': '0',
 'tokens': ['@paulwalk',
  'It',
  "'s",
  'the',
  'view',
  'from',
  'where',
  'I',
  "'m",
  'living',
  'for',
  'two',
  'weeks',
  '.',
  'Empire',
  'State',
  'Building',
  '=',
  'ESB',
  '.',
  'Pretty',
  'bad',
  'storm',
  'here',
  'last',
  'evening',
  '.'],
 'ner_tags': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  7,
  8,
  8,
  0,
  7,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

In [6]:
# Print the list of NER tags available in the dataset
# The position of each tag gives us the int values of NER tags in the dataset.
label_list = wnut["train"].features[f"ner_tags"].feature.names
print('Available NER tags:', label_list)

Available NER tags: ['O', 'B-corporation', 'I-corporation', 'B-creative-work', 'I-creative-work', 'B-group', 'I-group', 'B-location', 'I-location', 'B-person', 'I-person', 'B-product', 'I-product']


In [7]:
# The number of available NER tags is the number of labels needed in our classifier,
# i.e. the number of labels that our classifier will need to predict
num_labels = len(label_list)
print('Number of NER tags in the dataset:', num_labels)

Number of NER tags in the dataset: 13


To allow our model to process the dataset, we need to transform our input text into features that our network can understand.

In order to do this, we use a tokenizer that takes in input our text and returns the input ids, the token type ids and the attention mask associated to the given text.

In [8]:
# Load the pretrained BERT tokenizer from HuggingFace
tokenizer = BertTokenizerFast.from_pretrained(pretrained_model_name)

In [9]:
# Let's tokenize the first sample in the train set to check what happens when it's tokenized
tokenized_input = tokenizer(wnut['train'][0]["tokens"], is_split_into_words=True)

# Convert the tokenized input from token ids to string tokens
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])

In [10]:
print('Tokenized input:', tokenized_input)

print('\nInput:', wnut['train'][0]["tokens"])
print('Tokenized input ids:', tokenized_input['input_ids'])
print('Tokenized input tokens:', tokens)

print('\nLength of Input:', len(wnut['train'][0]["tokens"]))
print('Length of Tokenized input:', len(tokenized_input['input_ids']))
print('Length of Target labels:', len(wnut['train'][0]["ner_tags"]))

Tokenized input: {'input_ids': [101, 1030, 2703, 17122, 2009, 1005, 1055, 1996, 3193, 2013, 2073, 1045, 1005, 1049, 2542, 2005, 2048, 3134, 1012, 3400, 2110, 2311, 1027, 9686, 2497, 1012, 3492, 2919, 4040, 2182, 2197, 3944, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Input: ['@paulwalk', 'It', "'s", 'the', 'view', 'from', 'where', 'I', "'m", 'living', 'for', 'two', 'weeks', '.', 'Empire', 'State', 'Building', '=', 'ESB', '.', 'Pretty', 'bad', 'storm', 'here', 'last', 'evening', '.']
Tokenized input ids: [101, 1030, 2703, 17122, 2009, 1005, 1055, 1996, 3193, 2013, 2073, 1045, 1005, 1049, 2542, 2005, 2048, 3134, 1012, 3400, 2110, 2311, 1027, 9686, 2497, 1012, 3492, 2919, 4040, 2182, 2197, 3944, 1012, 102]
Tokenized input tokens: ['[CLS]', '@', 'paul', '##walk', 'it', "'", 's', '

As we can see, the length of our target labels and the tokenized input are different because BERT tokenizer might split input words into multiple subwords. However, the target labels correspond to entire words, not subwords. For this reason, it's crucial to align the target labels with the tokens produced by the tokenizer.


`tokenize_and_align_labels` function, iterates over each sample to align labels with the tokenized input.

`word_ids` function call retrieves the index of the original word each token belongs to, allowing for alignment between labels and tokens.
 - Tokens not associated with any word (like `[CLS]`, `[SEP]`, `[PAD]`) get a label of -100, indicating they should be ignored in loss computation.
 - For words split into multiple tokens, only the first token receives the original label. Subsequent tokens (subwords) are labeled with -100 to avoid double-counting the word's label.

In [11]:
def tokenize_and_align_labels(samples, tokenizer: BertTokenizerFast):
    """
    Tokenize the input samples using the given tokenizer and
    align the target labels with the tokens produced by the tokenizer.

    Parameters:
    - samples (): Input tensor of shape [batch_size, seq_len, embed_size].
    - tokenizer (BertTokenizerFast): tokenizer used to tokenize the data

    Returns:
    - DatasetDict: Output the dataset containing the aligned target labels.
    """

    # Tokenize the tokens in the dataset
    tokenized_inputs = tokenizer(samples["tokens"], truncation=True, is_split_into_words=True)

    labels = [] # list of new labels
    for i, ner_tags in enumerate(samples[f"ner_tags"]):
        # Map each token in the tokenized input to their respective word in the original sample words,
        # i.e. you obtain a list containing for each output token the index of the associated original word.
        # If the index is None, it means a special token has been added to the input.
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None  # index of the last checked word
        label_ids = []
        for word_idx in word_ids:
            # Set the special tokens to -100 to be ignored in loss computation.
            if word_idx is None:
                label_ids.append(-100)
            # Only label the first token of a given word.
            # If the previous checked word is different than the current word
            elif word_idx != previous_word_idx:
                label_ids.append(ner_tags[word_idx])
            else:
                label_ids.append(-100)
            # Updated the id of the last checked word
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [12]:
# Map the dataset to tokens and aligned labels using the tokenize_and_align_labels function
# batched=True is used to speed up the map function by processing multiple elements of the dataset at once
# fn_kwargs is used to pass multiple parameters to the tokenize_and_align_labels function
tokenized_wnut = wnut.map(tokenize_and_align_labels, batched=True, fn_kwargs={'tokenizer': tokenizer})

In [13]:
# Dataset before mapping and target label alignment
wnut

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1287
    })
})

In [14]:
# Dataset after mapping and alignment
tokenized_wnut

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1287
    })
})

After the mapping, we can see that the mapped dataset contains more columns than the original dataset, as we added the tokenized input, the token type ids, the attention mask and the aligned labels for each sample in the dataset.

## 3 Low-Rank Adaptation (LoRA)

LoRA [(Hu et al., 2021)](http://arxiv.org/abs/2106.09685) adapts PLMs using low-rank decomposition matrices. The idea is that the update
of model parameters can be approximated using
low-dimensional decomposition. LoRA reparameterises the Attention queries and values weights
into low-rank matrices. For each, LoRA uses two
small linear projection layers
to reparameterise the weights. LoRA receives the
same input that the reparameterised weights receive
(i.e. the insertion form is parallel).



### 3.1 LoRA Architecture

In [15]:
class LoRA(nn.Module):
    def __init__(self, input_dim, output_dim, rank):
        """
        Parameters:
        - input_dim (int): Input dimension size (features size), in our case that's the model dimension.
        - output_dim (int): Output dimension size (features size), in our case that's also the model dimension.
        - rank (int): Between the input_dim and output_dim, we can project the features size to a lower dimension and restore it again
                      input_dim -> rank -> output_dim.
                      Rank for the low-rank matrices A and B (trainable matrices used to approximate the orignal matrix). This is a hyperparameter that controls
                      the approximation's quality and the number of additional parameters introduced.
                      Lower ranks result in fewer parameters and vice versa.

        Attributes:
        - A (torch.nn.Parameter): A learnable parameter matrix of shape (in_dim, rank). Initialised using
                                  random Gaussian distribution.
                                  This matrix is part of the low-rank approximation.
        - B (torch.nn.Parameter): A learnable parameter matrix of shape (rank, out_dim). Initialised as
                                  zeros, which will be updated during training. This matrix is the second
                                  part of the low-rank approximation.
        """
        super().__init__()
        # We will use LoRA paper (https://arxiv.org/abs/2106.09, Section 4.1) recommended initialisation for A and B:
        # A: Random Gaussian initialisation
        # B: Initialised with Zero
        # In the LoRA implementaiton the initialisation of A, B are flipped, (Mohammed Sabry: mhmsabry) raised an issue
        # at https://github.com/microsoft/LoRA/issues/42, author said that it won't make a difference.
        # However, here we will stick ot the paper initialisation.

        # torch.randn already samples from a Gaussain distribution of mean 0 and std 1
        # https://pytorch.org/docs/stable/generated/torch.randn.html
        self.A = nn.Parameter(torch.randn(input_dim, rank))

        # torch.zeros will give us zero matrix
        self.B = nn.Parameter(torch.zeros(rank, output_dim))


    def forward(self, x):
        """
        Define the forward pass for the LoRA layer.

        Parameters:
        - x (torch.Tensor): Input tensor to the layer. The last dimension of this tensor should match `input_dim`.

        Returns:
        - torch.Tensor: The transformed tensor after applying the LoRA adjustments.
                        This involves matrix multiplication of the input tensor with the low-rank matrices A and B.
        """

        # In project_down, we go from input_dim (features size) to a small dim (Rank)
        project_down = x @ self.A
        # In project_up, we go from a small dim (Rank) to output_dim (features size)
        project_up = project_down @ self.B

        # Note: Majority of PEFT techniques follows this path of scaling down and then up

        return project_up

### 3.2 LoRA Collaboration With other Layers

Before we start implementing how LoRA collaborate with other layers, we need to remember the following facts:

1. LoRA only collaborate with Linear layers.
2. LoRA is inserted in parallel (i.e takes the linear layers input and process it in its own, and doesn't wait for linear layers output).
3. LoRA adds the output of its transformation to the output of the linear layers.

In [16]:
class LayerWithLoRA(nn.Module):
    def __init__(self, linear: nn.Linear, rank: int = 8, alpha: float = 16.0):
        """
        Initialise the LayerWithLoRA module, combining a standard linear layer with a LoRA layer.

        Steps:
        1. LoRA only collaborates with Linear layers: This class is specifically designed to work
           with an existing linear layer, enhancing its capabilities with the LoRA technique.

        Parameters:
        - linear (torch.nn.Linear): The linear layer to which LoRA will be applied. This layer should
                                    be an instance of torch.nn.Linear, indicating LoRA's compatibility
                                    strictly with linear layers.
        - rank (int): The rank used for the LoRA layer, determining the size of the low-rank matrices
                      and controlling the approximation's granularity.
        - alpha (float): Scaling factor for the LoRA adjustment. This controls how much the LoRA-adjusted
                         layer contributes to the final output, in conjunction with the original layer weights.

        Attributes:
        - linear (torch.nn.Linear): The original linear layer to which LoRA adjustments will be applied.
        - lora (LoRA): The LoRA layer initialised with dimensions matching the input and output
                            features of the provided linear layer, and configured with the specified rank.
                            It operates in parallel to the linear layer.
        """
        super().__init__()
        self.linear = linear  # Store the provided linear layer.
        # Initialise the LoRA layer in parallel to the existing linear layer, ensuring it operates
        # on the same input and output dimensions. This parallel operation allows LoRA to complement
        # the linear layer without waiting for its output.
        self.lora = LoRA(
            linear.in_features, linear.out_features, rank
        )

        # to scale LoRA transformation to balance its contribution.
        self.alpha = alpha

    def forward(self, x):
        """
        Defines the forward pass through the LinearWithLoRA module.

        Steps:
        2. LoRA is inserted in parallel: The input x is processed by both the linear layer and the LoRA
           layer independently. This parallel processing allows the LoRA layer to apply its adjustments
           without needing to wait for the output of the linear layer.

        3. LoRA adds the output of its transformation to the output of the linear layers: The final
           output of this module is the sum of the outputs from both the linear layer and the LoRA layer,
           effectively combining their transformations.

        Parameters:
        - x (torch.Tensor): The input tensor to the module.

        Returns:
        - torch.Tensor: The output tensor, which is the sum of the outputs from the linear layer and the
                        LoRA layer, demonstrating how LoRA enhances the original linear transformation.
        """
        # Process input through the original linear layer.
        linear_output = self.linear(x)

        # Process input in parallel through the LoRA layer.
        lora_output = self.lora(x)

        # Sum the outputs from both the linear and scaled LoRA layers with alpha, and return the combined result.
        return linear_output + self.alpha * lora_output


## 4 Adapters

Adapters [(Houlsby et al., 2019)](https://proceedings.mlr.press/v97/houlsby19a.html) use a feedforward layer that bottlenecks
information via two linear layers that project
the information down and then up, with ReLU
activation in-between. Adapters adapt the hidden
representations resulting from Attention and FNN
blocks (insertion form = sequential).

### 4.1 Adapters Architecture

In [17]:
class Adapters(nn.Module):
    def __init__(self, module_output_dim, bottleneck_dim):
        """
        Parameters:
        - module_output_dim (int): Output dimension size (features size) from the previous module, in our case that's model dimension.
        - bottleneck_dim (int): Between the input_dim and output_dim, we can project the features size to a lower dimension and restore it again
                      input_dim -> bottleneck_dim -> output_dim.
                      it's similar to rank in LoRA.

        Attributes:
        - W1 (torch.nn.Linear): A linear layer of shape (module_output_dim, bottleneck_dim).
        - W2 (torch.nn.Linear): A linear layer of shape (bottleneck_dim, module_output_dim).
        """
        super().__init__()

        self.W1 = nn.Linear(module_output_dim, bottleneck_dim)

        self.W2 = nn.Linear(bottleneck_dim, module_output_dim)


    def forward(self, x):
        """
        Define the forward pass for the Adapter layer.

        Parameters:
        - x (torch.Tensor): Input tensor to the layer. The last dimension of this tensor should match `input_dim`.

        Returns:
        - torch.Tensor: The transformed tensor after applying the Adapters layers.
        """

        # In project_down, we go from input_dim (features size) to a small dim (bottleneck_dim)
        project_down = self.W1(x)

        # In project_up, we go from a small dim (bottleneck_dim) to output_dim (features size)
        # Apply ReLU activations in between
        project_up = self.W2(F.relu(project_down))

        # Note: Majority of PEFT techniques follows this path of scaling down and then up

        return project_up

### 4.2 Adapters Collaboration with other Layers

Before we start implementing how Adapters collaborate with other layers, we need to remember the following facts:

1. Adapters don't necessary requires the layers they are collaborating with to be linear layers, becuase, they take whatever that layer produces and apply transformation on it.
2. Adapters are inserted sequentially (i.e takes the output of the previous layer-collaboration layer- process it and feed it to the next layer).
3. Before passing its ouput to the next layer, Adapters add its output to collaboration layer-previous layer- output and pass this accumulation.


In [18]:
class LayerWithAdapters(nn.Module):
    def __init__(self, module_projection: nn.Linear, bottleneck_dim: int = 16):
        """
        Initialise the LayerWithAdapters module, combining the output of Attention FeedForward layers with Adapter layer.

        Steps:
        1. Adapters don't necessary requires the layers they are collaborating with to be linear layers,
         becuase, they take whatever that layer produces and applying transformation on it.

        Parameters:
        - module_projection (torch.nn.Linear): It's the last layer in the module that gives the output of the module, which we want the Adapters layer to consume.
                                               and in most cases, it's a linear layer.
        - bottleneck_dim (int): Between the input_dim and output_dim, we can project the features size to a lower dimension and restore it again
                      input_dim -> bottleneck_dim -> output_dim.
                      it's similar to rank in LoRA.

        Attributes:
        - module_projection (torch.nn.Linear): The original linear layer to which LoRA adjustments will be applied.
        - adapter_layers (Adapters): The Adapters layer initialised with dimensions matching the input and output
                            features of the provided module_projection layer, and configured with the specified bottleneck_dim
                            . It operates sequentially after to the module_projection layer.
        """
        super().__init__()
        self.module_projection = module_projection  # Store the provided module_projection layer.
        # Initialise the Adapters layer.
        self.adapter_layers = Adapters(
             module_projection.out_features, bottleneck_dim
        )



    def forward(self, x):
        """
        Defines the forward pass through the LinearWithLoRA module.

        Steps:
        2. Adapters are inserted sequentially(i.e takes the output of the previous
              layer-collaboration layer- process it and feed it to the next layer).

        3. Before passing its ouput to the next layer,
            Adapters add its output to collaboration layer-previous layer- output
            and pass this accumulation.

        Parameters:
        - x (torch.Tensor): The input tensor to the module.

        Returns:
        - torch.Tensor: The output tensor, which is the sum of the outputs from the module layer and the
                        Adapters layer.
        """
        # Process input through the original module projection layer.
        module_output = self.module_projection(x)

        # Process the output of module projection layer with Adapters layer.
        adapters_output = self.adapter_layers(module_output)

        # Sum the outputs from both the Adapter layers and Module output, and return the combined result.
        return module_output + adapters_output


## 5 Training with HugginFace API

### 5.1 Dataset Class

HuggingFace will create a torch Dataset class automatically, we just need to pass the datasets to a class called `Trainer` (Section 4.4).


**Note:** We need to ensure that our datasets contains columns with names like `input_ids` and `labels` as HuggingFace codebase expects the dataset contains to these columns.



In [19]:
train_dataset_hf, eval_dataset_hf, test_dataset_hf = tokenized_wnut["train"], tokenized_wnut["validation"], tokenized_wnut["test"]

HuggingFace's APIs, similar to PyTorch, enable you to utilise a Data Collator class to customise the batching of your inputs according to your requirements.


HuggingFace has created many Data Collator classes for different tasks, here we will use `DataCollatorForTokenClassification` class, as it suited for our task.

The [DataCollatorForTokenClassification](https://huggingface.co/docs/transformers/v4.17.0/en/main_classes/data_collator#transformers.DataCollatorForTokenClassification) creates a batch of examples, dynamically padding the text and labels to the length of the longest element in the batch, so they are a uniform length. This is more efficient than pad the text in the tokenizer function.

In [20]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

### 5.2 Initialise our Model with a PEFT

after constructing LoRA/Adapters architecture, processing mode (i.e parallel/sequential) and their collaboration with other layers, now we will attach them to a BERT model.

In [21]:
# Let's initialise our classification model
# we will wrap the model initialsation on a function as we want to use the model with different PEFT each time
def get_BERT():
    bert_model = BertModel.from_pretrained(pretrained_model_name) # Load a pretrained BERT model
    num_labels = len(wnut["train"].features[f"ner_tags"].feature.names)  # Update this based on your NER task
    model = BertClassifier(bert_model, num_labels=num_labels).to(device)
    return model  # init our classifier model

#### 5.2.1 BERT Model before PEFT

In [22]:
get_BERT()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_af

#### 5.2.2 BERT Model after PEFT


LoRA paper conducted ablation experimenets of which linear layers give the best performance, and they found attaching LoRA transformation to queries and keys give the best performance.

Adapters paper attaches the layers after the attention and the feedforward blocks.

In [23]:
# freeze the pretrained models layer, however, we will let the classification layers to be trained with LoRA or Adapters layers
# As it's randomly initialised
def disable_grads(model):
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False

In [24]:
rank = 8
alpha = 16.0
def attach_LoRA():
  global model
  model = get_BERT()
  disable_grads(model) #disable model's gradient except the classifier layer
  for layer in model.bert.encoder.layer:
      layer.attention.self.query = LayerWithLoRA(layer.attention.self.query, rank=rank, alpha=alpha)
      layer.attention.self.key = LayerWithLoRA(layer.attention.self.key, rank=rank, alpha=alpha)


bottleneck_dim = 16
def attach_Adapters():
  global model
  model = get_BERT()
  disable_grads(model) #Disable model's gradient except the classifier layer
  for layer in model.bert.encoder.layer:
      layer.attention.output.dense = LayerWithAdapters(layer.attention.output.dense, bottleneck_dim)
      layer.output.dense = LayerWithAdapters(layer.output.dense, bottleneck_dim)

In [25]:
# @title Select a PEFT to attach to BERT

dropdown = widgets.Dropdown(
    options=[('Select an option', None), ('LoRA', 1), ('Adapters', 2)],
    value=None,
    description='Select PEFT:',
)

def on_dropdown_change(change):
    if change['new'] == 1:
        attach_LoRA()
    elif change['new'] == 2:
        attach_Adapters()

dropdown.observe(on_dropdown_change, names='value')
display(dropdown)

Dropdown(description='Select PEFT:', options=(('Select an option', None), ('LoRA', 1), ('Adapters', 2)), value…

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [27]:
#check the parameters that requires grad/update:
for name, param in model.named_parameters():
  if param.requires_grad:
    print(name)

bert.encoder.layer.0.attention.self.query.lora.A
bert.encoder.layer.0.attention.self.query.lora.B
bert.encoder.layer.0.attention.self.key.lora.A
bert.encoder.layer.0.attention.self.key.lora.B
bert.encoder.layer.1.attention.self.query.lora.A
bert.encoder.layer.1.attention.self.query.lora.B
bert.encoder.layer.1.attention.self.key.lora.A
bert.encoder.layer.1.attention.self.key.lora.B
bert.encoder.layer.2.attention.self.query.lora.A
bert.encoder.layer.2.attention.self.query.lora.B
bert.encoder.layer.2.attention.self.key.lora.A
bert.encoder.layer.2.attention.self.key.lora.B
bert.encoder.layer.3.attention.self.query.lora.A
bert.encoder.layer.3.attention.self.query.lora.B
bert.encoder.layer.3.attention.self.key.lora.A
bert.encoder.layer.3.attention.self.key.lora.B
bert.encoder.layer.4.attention.self.query.lora.A
bert.encoder.layer.4.attention.self.query.lora.B
bert.encoder.layer.4.attention.self.key.lora.A
bert.encoder.layer.4.attention.self.key.lora.B
bert.encoder.layer.5.attention.self.quer

### 5.3 TrainingArguments

**TrainingArguments** provides a structured way to configure training hyperparameters, such as learning rate, batch size, number of epochs, logging behaviour, and much more.

For more details check [here](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)

In [28]:
training_args = TrainingArguments(
    output_dir='./model_output',    # Directory where model checkpoints and outputs will be saved.
    num_train_epochs=3,# Total number of training epochs.
    per_device_train_batch_size=16, # Batch size per device during training.
    per_device_eval_batch_size=64,  # Batch size for evaluation.
    learning_rate=5e-5,             # Learning rate
    warmup_steps=500,               # Number of warmup steps for learning rate scheduler.
    weight_decay=0.01,              # Weight decay if we apply some.
    logging_dir='./logs',           # Directory for storing logs.
    logging_steps=10,               # Log every X updates steps.
    eval_strategy="steps",    # Evaluate every X steps.
    eval_steps=50,                  # Number of steps to evaluate after.
    save_strategy="steps",          # The checkpoint save strategy to use.
    save_steps=100,                 # Save checkpoint every X steps.
    load_best_model_at_end=True,    # Whether to load the best model found at each evaluation.
    report_to="none",                # The list of integrations to report the results and logs to.
    metric_for_best_model="f1",       # use F1 to decide what "best" means
    greater_is_better=True           # higher F1 is better (opposite of loss)
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### 5.4 compute_metrics

`compute_metrics` is a function used with Hugging Face's Trainer class to evaluate the performance of a model on a given dataset using metrics such as accuracy, precision, recall, F1 score. It's only used during evaluation, mostly practitioners use it to save the best model during training, and to track the important metrics during the training process.



In [29]:
metric = evaluate.load("seqeval")

def compute_metrics(model_output):
    """
    Compute evaluation metrics to check the performance of the model during training (on the validation set)
    and after training (on the test set).
    The input parameter of the function is a Tuple as required by the HuggingFace Trainer.

    Parameters:
    - model_output (Tuple): Contains model's raw predictions and target labels.

    Returns:
    - dict: Dictionary of the evaluation metrics, i.e. Precision, Recall, F1, and Accuracy.
    """
    # 'model_output' contains the model's raw predictions and the true labels for the evaluation dataset.
    predictions, labels = model_output

    # Convert logits to predicted class indices by selecting the index of the max logit in each token position.
    predictions = np.argmax(predictions, axis=2)

    # Filter out the special tokens and align the predictions with the actual labels.
    # This step is necessary because models like BERT use special tokens (e.g., [CLS], [SEP], [PAD]),
    # and labels for these tokens are typically set to an ignore index (e.g., -100) so they don't affect loss computation.
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Compute evaluation metrics such as precision, recall, F1 score, and accuracy using the 'metric' object.
    # This 'metric' object is typically an instance from the Hugging Face's `datasets` library, which provides
    # various metrics calculation functions.
    results = metric.compute(predictions=true_predictions, references=true_labels)

    # Return a dictionary containing the computed metrics.
    return {
        "precision": results["overall_precision"],  # The proportion of true positive results in all positive predictions.
        "recall": results["overall_recall"],        # The proportion of true positive results in all actual positives.
        "f1": results["overall_f1"],                # The harmonic mean of precision and recall.
        "accuracy": results["overall_accuracy"],    # The proportion of true results (both true positives and true negatives) in the dataset.
    }


### 5.5 Trainer

The Trainer class in Hugging Face's Transformers library is designed to simplify the training, evaluation, and testing of transformer models by providing a simple API where you only need to provide your model, training and evaluating sets, training arguments and an optional compute metrics function.

For more details, check [here](https://huggingface.co/docs/transformers/main_classes/trainer#trainer)

In [30]:
trainer = Trainer(
    model=model,                         # The model to be trained or fine-tuned.
    args=training_args,                  # TrainingArguments object containing the training and evaluation configurations.
    train_dataset=train_dataset_hf,         # The dataset to be used during training. Should be a Hugging Face Dataset or a dataset that implements __len__ and __getitem__.
    eval_dataset=eval_dataset_hf,           # The dataset for evaluation. Similar format as train_dataset. Used to evaluate the model performance at each logging step or epoch end.
    compute_metrics=compute_metrics,     # A function that computes metrics of interest for evaluation. It takes an EvalPrediction object (which has .predictions and .label_ids attributes) and should return a dictionary mapping metric names to their values.
    data_collator=data_collator          # Ensures samples are properly padded and batched
    )


### 5.6 Calling train()

In [31]:
# After initializing our training pipeline with Trainer, we can start training
# by simply using the train function
# train() will train the specified model using the train_dataset given in input in the initialisation,
# evaluate the trained network every x number of steps using the validation set (eval_dataset)
# and save/load the best model at the end of the training
train_output = trainer.train()

/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
50,2.114853,2.150846,0.008751,0.065789,0.015447,0.510141
100,1.178942,1.393341,0.027397,0.004785,0.008147,0.914235
150,0.586035,0.745411,0.000000,0.000000,0.000000,0.920529
200,0.395013,0.506214,0.000000,0.000000,0.000000,0.920529
250,0.367504,0.462330,0.000000,0.000000,0.000000,0.920529
300,0.298189,0.448819,0.000000,0.000000,0.000000,0.920529
350,0.318252,0.424636,0.000000,0.000000,0.000000,0.920529
400,0.239089,0.421775,0.000000,0.000000,0.000000,0.920529
450,0.292558,0.371592,0.500000,0.001196,0.002387,0.920656
500,0.305805,0.361198,0.750000,0.007177,0.014218,0.921038


/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/robertsparks/Documents/neural_langu

### 5.7 Calling evaluate()

In [32]:
# The notebook progress callback can raise
# `RuntimeError: on_train_begin must be called before on_evaluate`
# when we call `trainer.evaluate()` after training in a notebook/Colab.
# This callback only affects the progress-display UI, not the model,
# predictions, or metric computation, so we remove it before evaluation.
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

print('Evaluation metrics on the validation set:')
trainer.evaluate()

Evaluation metrics on the validation set:


/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.359269380569458,
 'eval_precision': 0.45901639344262296,
 'eval_recall': 0.03349282296650718,
 'eval_f1': 0.06243032329988852,
 'eval_accuracy': 0.9226905715557251,
 'eval_runtime': 2.5986,
 'eval_samples_per_second': 388.28,
 'eval_steps_per_second': 6.157,
 'epoch': 3.0}

### 5.8 Calling predict()

At this point, we want to test how good our model is, so we need to use our trained model to make predictions on the test set.

At the end of training, the Trainer class loaded the best trained model in the pipeline. We simply use the predict function passing the test set to compute the predictions.

In [33]:
predictions = trainer.predict(test_dataset_hf)
print('Evaluation metrics on the test set:')
# predict() return multiple information, such as predicted logits for each sample and
# evaluation metrics computed using the compute_metrics function.
# In this moment, we're interested in accessing the evaluation metric to evaluate how good our network is
predictions.metrics

Evaluation metrics on the test set:


{'test_loss': 0.37263011932373047,
 'test_precision': 0.29347826086956524,
 'test_recall': 0.025023169601482854,
 'test_f1': 0.04611443210930828,
 'test_accuracy': 0.9272797229703732,
 'test_runtime': 5.8427,
 'test_samples_per_second': 220.275,
 'test_steps_per_second': 3.594}



---



# Week 8 Submission Task

Apply PEFT to the model and compare the obtained results with the results obtained using complete fine-tuning (Week 7 lab).

Compare the results with the full fine-tuning
